In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd

from automed import *
from IPython.display import display, Markdown as IMarkdown
from rich.console import Console
from rich.markdown import Markdown

C:\Users\0939300\AppData\Local\Temp\ipykernel_8100\2150211869.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


[16:59:27] cuDF not found: falling back to standalone pandas.

In [2]:
#titanic = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')
titanic = pd.read_csv('../perf_logger/tests_data/IMDB-Dataset.csv', delimiter=',')[:10]

In [3]:
titanic.shape

(10, 2)

In [4]:
autom = AutoMed()

print(autom.default_pipeline())
print(autom.json_pipeline())

None
{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDateConverter', 'name': 'Convert Short text to date if possible', 'description': 'Step description...', 'configuration': {'authorized_error_ratios': {'default': 0.05, 'description': 'Over this ratios, the column will not be converted into date', 'value': 0.05}, 'sample_size': {'default': 200, 'description': 'Convert date is time consuming.                     To save time, date detection will be done on a random sample.                     Set to -1 to detect on the whole dataset', 'value': 200}}, 'children': []}]}, {'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActSplitDate', 'name': 'Transform string column to date', 'description': 'Step description...', 'configu

In [5]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [
        {
            'step': 'ActDropNumericalColumn',
            'configuration': {
                'empty_threshold': { 'value': 0.1 },
            }
        },
        {
            'step': 'MetaStep',
            'tag': 'cleaning',
        },
        #{
        #    'step': 'MetaStep',
        #    'tag': 'features_selection',
        #},
        #{
        #   'step': 'WrapKFold',
        #   'children': [{
        #       'step': 'ActKNN',
        #   }]
        #},
    ]
}

autom.load_pipeline(pipeline)
print(autom.json_pipeline())


{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropNumericalColumn', 'name': 'Drop numerical columns', 'description': 'Drop numerical columns where the proportion of empty rows\n        in the dataset is higher than {empty_threshold}.', 'configuration': {'empty_threshold': {'description': 'Column with more or equal proportion of empty row                     will dropped. 1 will drop all columns', 'default': 0.5, 'value': 0.1}}, 'children': []}, {'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActSplitDate', 'name': 'Transform string column to date', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActMeanColumn', 'name': 'Fill missing values with mean', 'description': 'Fills missing values with the mean of non-missing values\n        when the proportion of empty rows is lower than {empty_thr

In [6]:
# results = autom.fit(
#     titanic.drop('label', axis=1).copy(),
#     titanic[['label']]).copy()
results = autom.fit(
    titanic.drop('sentiment', axis=1).copy(),
    titanic[['sentiment']]).copy()

c:\Users\0939300\Desktop\Automed\.venv\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[16:59:28] running step: MetaOrderedStep (steps=ActDropNumericalColumn,MetaStep)

           running step: MetaStep                                                                                  
           (steps=ActDropTextualColumn,ActSplitDate,ActMeanColumn,ActDropNumericalColumn,ActOnehot,ActCategoryStrin
           gToNumeric,ActDropDateColumn,ActWord2Vec)

c:\Users\0939300\Desktop\Automed\.venv\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

IndexError: list index out of range

In [ ]:
print(results[0].pipeline.model)
print(results[0].pipeline.steps)

console = Console()

for step in results[0].pipeline.explanations:
    md = step.to_markdown()
    if md:
        # console.print(Markdown(md))
        display(IMarkdown(md))

pm = results[0].pipeline.pickle()

In [ ]:
import pickle

o = 200 # offset
n = 68  # # of samples
labels = titanic.iloc[o:(o+n)]['label']
predict_df = titanic.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)
sum([ r == labels[o+i] for i, r in enumerate(m.predict(predict_df)) ]) / n

In [ ]:
final_boss_automed = AutoMed(max_workers=2)
final_boss_automed.default_pipeline()
final_boss_results = final_boss_automed.fit(titanic.drop('label', axis=1).copy(), titanic[['label']].copy())

In [ ]:
# sum([ len(r.model.pickle()) for r in final_boss_results ])
[ (r.model.ml_model, r.evaluate()) for r in final_boss_results if r.model.ml_model is not None ]